In [3]:
!pip install pyspark -q
!wget -q https://s3.amazonaws.com/redshift-downloads/drivers/jdbc/2.1.0.30/redshift-jdbc42-2.1.0.30.jar

import os
print("PySpark e driver JDBC prontos!" if os.path.exists("redshift-jdbc42-2.1.0.30.jar") else "Erro no download")

PySpark e driver JDBC prontos!


In [7]:
REDSHIFT_HOST = "default-workgroup.381492065649.us-east-2.redshift-serverless.amazonaws.com"
REDSHIFT_PORT = "5439"
REDSHIFT_DB   = "dev"
REDSHIFT_USER = "fraudeuser"
REDSHIFT_PASS = "Fraude#2024!"

JDBC_URL = f"jdbc:redshift://{REDSHIFT_HOST}:{REDSHIFT_PORT}/{REDSHIFT_DB}"
print(f"JDBC URL: {JDBC_URL}")

JDBC URL: jdbc:redshift://default-workgroup.381492065649.us-east-2.redshift-serverless.amazonaws.com:5439/dev


In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PicPay-FraudeSentinel-Redshift") \
    .config("spark.jars", "redshift-jdbc42-2.1.0.30.jar") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"Spark pronto! Versão: {spark.version}")

Spark pronto! Versão: 4.0.2


In [9]:
# Propriedades de conexão JDBC
jdbc_props = {
    "user": REDSHIFT_USER,
    "password": REDSHIFT_PASS,
    "driver": "com.amazon.redshift.jdbc42.Driver"
}

# Teste simples — lê as tabelas existentes no Redshift
try:
    df_teste = spark.read.jdbc(
        url=JDBC_URL,
        table="(SELECT current_date AS hoje, current_user AS usuario) AS teste",
        properties=jdbc_props
    )
    df_teste.show()
    print("Conexão com Redshift estabelecida com sucesso!")
except Exception as e:
    print(f"Erro na conexão: {e}")

+----------+--------------------+
|      hoje|             usuario|
+----------+--------------------+
|2026-03-22|fraudeuser       ...|
+----------+--------------------+

Conexão com Redshift estabelecida com sucesso!


In [10]:
# Propriedades de conexão JDBC
jdbc_props = {
    "user": REDSHIFT_USER,
    "password": REDSHIFT_PASS,
    "driver": "com.amazon.redshift.jdbc42.Driver"
}

# Teste simples — lê as tabelas existentes no Redshift
try:
    df_teste = spark.read.jdbc(
        url=JDBC_URL,
        table="(SELECT current_date AS hoje, current_user AS usuario) AS teste",
        properties=jdbc_props
    )
    df_teste.show()
    print("Conexão com Redshift estabelecida com sucesso!")
except Exception as e:
    print(f"Erro na conexão: {e}")

+----------+--------------------+
|      hoje|             usuario|
+----------+--------------------+
|2026-03-22|fraudeuser       ...|
+----------+--------------------+

Conexão com Redshift estabelecida com sucesso!


In [12]:
from pyspark.sql.functions import col, when, count, avg
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

# Relendo o CSV original
df = spark.read.csv(
    "/content/PS_20174392719_1491204439457_log.csv",
    header=True,
    inferSchema=True
)

# Refazendo filtro e features
df_spark = df.filter(col("type").isin(["CASH_OUT", "TRANSFER"])) \
             .withColumn(
                 "erro_saldo_orig",
                 col("newbalanceOrig") - (col("oldbalanceOrg") - col("amount"))
             )

print(f"Dados carregados: {df_spark.count():,} transações")

Dados carregados: 61,709 transações


In [13]:
# Selecionando colunas para salvar no Redshift
df_redshift = df_spark.select(
    "type",
    "amount",
    "nameOrig",
    "nameDest",
    "oldbalanceOrg",
    "newbalanceOrig",
    "erro_saldo_orig",
    "isFraud",
    "isFlaggedFraud"
)

print("Escrevendo no Redshift... (pode levar alguns minutos)")

df_redshift.write \
    .format("jdbc") \
    .option("url", JDBC_URL) \
    .option("dbtable", "public.transacoes_fraude") \
    .option("user", REDSHIFT_USER) \
    .option("password", REDSHIFT_PASS) \
    .option("driver", "com.amazon.redshift.jdbc42.Driver") \
    .option("batchsize", "10000") \
    .mode("overwrite") \
    .save()

print("Dados escritos no Redshift com sucesso!")

Escrevendo no Redshift... (pode levar alguns minutos)
Dados escritos no Redshift com sucesso!


In [14]:
# Lendo de volta do Redshift para confirmar
df_confirmacao = spark.read \
    .format("jdbc") \
    .option("url", JDBC_URL) \
    .option("dbtable", "public.transacoes_fraude") \
    .option("user", REDSHIFT_USER) \
    .option("password", REDSHIFT_PASS) \
    .option("driver", "com.amazon.redshift.jdbc42.Driver") \
    .load()

print(f"Total de registros no Redshift: {df_confirmacao.count():,}")
print(f"Fraudes: {df_confirmacao.filter(col('isFraud')==1).count():,}")
df_confirmacao.show(5)

Total de registros no Redshift: 61,709
Fraudes: 131
+--------+----------+-----------+-----------+-------------+--------------+------------------+-------+--------------+
|    type|    amount|   nameorig|   namedest|oldbalanceorg|newbalanceorig|   erro_saldo_orig|isfraud|isflaggedfraud|
+--------+----------+-----------+-----------+-------------+--------------+------------------+-------+--------------+
|CASH_OUT| 212228.35|C1896074070| C401424608|          0.0|           0.0|         212228.35|      0|             0|
|TRANSFER|2545478.01|C1057507014|C1590550415|          0.0|           0.0|        2545478.01|      0|             0|
|CASH_OUT| 128278.74| C407493402|C1262822392|          0.0|           0.0|         128278.74|      0|             0|
|TRANSFER| 288638.46| C596333086| C747464370|          0.0|           0.0|         288638.46|      0|             0|
|CASH_OUT| 146534.96| C333879495|C1112414583|      21114.0|           0.0|125420.95999999999|      0|             0|
+--------+--